# Interpolate the Missing Days in 1-km HRDPS

## List of Missing days

`02Jun23` in /results/forcing/atmospheric/GEM1.0/GRIB

## Solution 

1. Use `/ocean/jqiu/forcings/HRDPS_1km` for 1-km HRDPS Grid;

2. Use `/results/forcing/atmospheric/continental2.5/` for 2.5-km HRDPS data;

3. Interpolate 2.5-km data into 1-km grids.

In [1]:
import xarray as xr

path_hrdps_25='/results/forcing/atmospheric/continental2.5/nemo_forcing/hrdps_y2023m06d02.nc'

ds_hrdps_25=xr.load_dataset(path_hrdps_25)

print(ds_hrdps_25)

<xarray.Dataset> Size: 55MB
Dimensions:           (time_counter: 24, y: 230, x: 190)
Coordinates:
  * time_counter      (time_counter) datetime64[ns] 192B 2023-06-02 ... 2023-...
  * y                 (y) int64 2kB 0 1 2 3 4 5 6 ... 224 225 226 227 228 229
  * x                 (x) int64 2kB 0 1 2 3 4 5 6 ... 184 185 186 187 188 189
    nav_lon           (y, x) float64 350kB 233.7 233.7 233.8 ... 239.1 239.1
    nav_lat           (y, x) float64 350kB 46.12 46.13 46.13 ... 51.77 51.77
Data variables:
    LHTFL_surface     (time_counter, y, x) float32 4MB 72.47 74.68 ... 56.7
    PRATE_surface     (time_counter, y, x) float32 4MB 0.0 0.0 0.0 ... 0.0 0.0
    RH_2maboveground  (time_counter, y, x) float32 4MB 70.12 69.38 ... 25.01
    atmpres           (time_counter, y, x) float32 4MB 1.021e+05 ... 1.01e+05
    precip            (time_counter, y, x) float32 4MB 0.0 0.0 0.0 ... 0.0 0.0
    qair              (time_counter, y, x) float32 4MB 0.005983 ... 0.004563
    solar             (time_c

## Interpolation

In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Interpolate HRDPS continental 2.5 km forcing onto the HRDPS 1 km grid.

Interpolation:
    - Inside 2.5 km convex hull:
        Delaunay triangulation + barycentric linear interpolation.
    - Outside 2.5 km convex hull:
        NaN. NO EXTRAPOLATION.

The interpolation geometry is calculated only once and reused for:
    - all hours
    - all variables
    - all requested days

After building the interpolation geometry, the script also checks:

    weights-HRDPS-1km_202108.nc

to determine whether any of the HRDPS 1 km source points referenced by
NEMO's src01 ... src04 interpolation stencil fall in the NaN region.

IMPORTANT:
    All four source indices are checked even if the corresponding
    interpolation weight is exactly zero, because IEEE arithmetic gives

        0.0 * NaN = NaN

    and therefore even a zero-weight NaN source point could potentially
    contaminate the model if all four stencil points are evaluated.

Requirements:
    numpy
    scipy
    netCDF4
"""

from pathlib import Path
from datetime import datetime

import numpy as np
from netCDF4 import Dataset
from scipy.spatial import Delaunay


# ============================================================
# USER SETTINGS
# ============================================================

# Genuine HRDPS 1 km forcing file used as the target-grid/template.
REFERENCE_1KM = Path(
    "/home/jqiu/analysis-junqi/"
    "Analysis_Atmospheric_Forcing/Data_Conversion/"
    "Data_HRDPS_1km_conversion/"
    "HRDPS_1km_y2023m03d01.nc"
)

# Existing HRDPS continental 2.5 km forcing.
SOURCE_25KM_DIR = Path(
    "/results/forcing/atmospheric/"
    "continental2.5/nemo_forcing"
)

# Existing HRDPS-1km -> NEMO interpolation weights.
NEMO_WEIGHTS_FILE = Path(
    "/home/jqiu/analysis-junqi/"
    "Analysis_Atmospheric_Forcing/Analysis_weights/"
    "HRDPS_1km_Weights/"
    "weights-HRDPS-1km_202108.nc"
)

# Temporary output directory.
OUTPUT_DIR = Path(
    "/home/jqiu/analysis-junqi/"
    "Analysis_Atmospheric_Forcing/Data_Conversion/"
    "Data_HRDPS_1km_conversion/"
    "from_2p5km_TEMP"
)

# ============================================================
# EDIT THE MISSING DAYS HERE
# ============================================================

DATES = [
    "2023-06-02",
    # "2023-06-03",
    # "2023-06-04",
]


VARIABLES = [
    "LHTFL_surface",
    "PRATE_surface",
    "RH_2maboveground",
    "atmpres",
    "precip",
    "qair",
    "solar",
    "tair",
    "therm_rad",
    "u_wind",
    "v_wind",
]


OVERWRITE = False

COMPRESSION_LEVEL = 4

# Used only for reporting whether a NaN source point has a non-zero
# interpolation weight.
WEIGHT_EPS = 1.0e-14

# Number of problematic stencil references to print individually.
MAX_BAD_EXAMPLES = 30


# ============================================================
# BASIC HELPERS
# ============================================================

def lon_to_180(lon):
    """Convert longitude to [-180, 180)."""
    return (np.asarray(lon, dtype=np.float64) + 180.0) % 360.0 - 180.0


def lonlat_to_xy(lon, lat, lat0):
    """
    Convert lon/lat to an approximately Cartesian coordinate system.

    Only used for constructing the regional Delaunay triangulation.
    """
    radius = 6371000.0

    lon = lon_to_180(lon)
    lat = np.asarray(lat, dtype=np.float64)

    x = (
        radius
        * np.deg2rad(lon)
        * np.cos(np.deg2rad(lat0))
    )

    y = radius * np.deg2rad(lat)

    return np.column_stack(
        (
            x.ravel(),
            y.ravel(),
        )
    )


def get_fill_value(var):
    """Return an existing NetCDF _FillValue, if present."""
    if "_FillValue" in var.ncattrs():
        return var.getncattr("_FillValue")

    return None


def copy_nc_attrs(src_var, dst_var, skip=None):
    """Copy NetCDF variable attributes."""
    if skip is None:
        skip = set()

    skip = set(skip)
    skip.add("_FillValue")

    for attr in src_var.ncattrs():

        if attr in skip:
            continue

        try:
            dst_var.setncattr(
                attr,
                src_var.getncattr(attr),
            )

        except Exception as exc:
            print(
                f"WARNING: could not copy attribute "
                f"{src_var.name}:{attr}: {exc}"
            )


# ============================================================
# FILE NAMES
# ============================================================

def make_source_filename(date_string):

    date = datetime.strptime(
        date_string,
        "%Y-%m-%d",
    )

    return SOURCE_25KM_DIR / (
        f"hrdps_y{date.year:04d}"
        f"m{date.month:02d}"
        f"d{date.day:02d}.nc"
    )


def make_output_filename(date_string):

    date = datetime.strptime(
        date_string,
        "%Y-%m-%d",
    )

    return OUTPUT_DIR / (
        f"HRDPS_1km_y{date.year:04d}"
        f"m{date.month:02d}"
        f"d{date.day:02d}.nc"
    )


# ============================================================
# BUILD INTERPOLATION GEOMETRY
# ============================================================

def create_interpolation_geometry(
    src_lon,
    src_lat,
    dst_lon,
    dst_lat,
):
    """
    Build Delaunay interpolation geometry.

    Only target points inside the source-grid convex hull receive
    interpolation weights.

    All target points outside the convex hull remain NaN.

    Returns
    -------
    inside_target_indices
        Flat indices into the HRDPS 1 km grid.

    vertices
        The three flattened HRDPS 2.5 km source indices used for
        each valid target point.

    weights
        Three barycentric weights for each valid target point.

    coverage_mask
        Boolean array with target-grid shape:
            True  = covered by 2.5 km interpolation
            False = output will be NaN
    """

    src_lon = np.asarray(
        src_lon,
        dtype=np.float64,
    )

    src_lat = np.asarray(
        src_lat,
        dtype=np.float64,
    )

    dst_lon = np.asarray(
        dst_lon,
        dtype=np.float64,
    )

    dst_lat = np.asarray(
        dst_lat,
        dtype=np.float64,
    )

    lat0 = np.nanmean(
        np.concatenate(
            (
                src_lat.ravel(),
                dst_lat.ravel(),
            )
        )
    )

    src_xy_all = lonlat_to_xy(
        src_lon,
        src_lat,
        lat0,
    )

    dst_xy_all = lonlat_to_xy(
        dst_lon,
        dst_lat,
        lat0,
    )

    src_valid = (
        np.isfinite(src_lon.ravel())
        & np.isfinite(src_lat.ravel())
    )

    dst_valid = (
        np.isfinite(dst_lon.ravel())
        & np.isfinite(dst_lat.ravel())
    )

    source_flat_indices = np.flatnonzero(
        src_valid
    )

    target_flat_indices = np.flatnonzero(
        dst_valid
    )

    src_xy = src_xy_all[src_valid]
    dst_xy = dst_xy_all[dst_valid]

    print()
    print("=" * 78)
    print("BUILDING 2.5 km -> 1 km INTERPOLATION GEOMETRY")
    print("=" * 78)

    print(
        f"Valid 2.5 km source points : "
        f"{len(src_xy):,}"
    )

    print(
        f"Valid 1 km target points   : "
        f"{len(dst_xy):,}"
    )

    print()
    print("Building Delaunay triangulation...")

    triangulation = Delaunay(
        src_xy
    )

    simplex = triangulation.find_simplex(
        dst_xy
    )

    inside_valid = simplex >= 0

    inside_target_indices = (
        target_flat_indices[inside_valid]
    )

    outside_target_indices = (
        target_flat_indices[~inside_valid]
    )

    n_inside = len(
        inside_target_indices
    )

    n_outside = len(
        outside_target_indices
    )

    n_bad_coord = (
        dst_lon.size
        - len(target_flat_indices)
    )

    print()
    print(
        f"Inside 2.5 km convex hull  : "
        f"{n_inside:,}"
    )

    print(
        f"Outside -> NaN             : "
        f"{n_outside:,}"
    )

    print(
        f"Invalid target coordinates : "
        f"{n_bad_coord:,}"
    )

    print(
        f"NaN fraction               : "
        f"{100.0 * (n_outside + n_bad_coord) / dst_lon.size:.4f}%"
    )

    # --------------------------------------------------------
    # Construct interpolation weights ONLY for inside points.
    # --------------------------------------------------------

    s = simplex[inside_valid]

    local_vertices = (
        triangulation.simplices[s]
    )

    vertices = source_flat_indices[
        local_vertices
    ]

    transform = triangulation.transform[s]

    delta = (
        dst_xy[inside_valid]
        - transform[:, 2, :]
    )

    bary12 = np.einsum(
        "nij,nj->ni",
        transform[:, :2, :],
        delta,
    )

    weights = np.empty(
        (n_inside, 3),
        dtype=np.float64,
    )

    weights[:, :2] = bary12

    weights[:, 2] = (
        1.0
        - bary12.sum(axis=1)
    )

    # --------------------------------------------------------
    # Coverage mask
    # --------------------------------------------------------

    coverage_flat = np.zeros(
        dst_lon.size,
        dtype=bool,
    )

    coverage_flat[
        inside_target_indices
    ] = True

    coverage_mask = coverage_flat.reshape(
        dst_lon.shape
    )

    # --------------------------------------------------------
    # Basic sanity check
    # --------------------------------------------------------

    weight_sum = weights.sum(
        axis=1
    )

    print()
    print("Barycentric-weight check:")

    print(
        "  min(sum(weights)) = "
        f"{weight_sum.min():.15f}"
    )

    print(
        "  max(sum(weights)) = "
        f"{weight_sum.max():.15f}"
    )

    print(
        "  min(weight)       = "
        f"{weights.min():.15e}"
    )

    print(
        "  max(weight)       = "
        f"{weights.max():.15e}"
    )

    if not np.allclose(
        weight_sum,
        1.0,
        atol=1.0e-10,
        rtol=0.0,
    ):
        raise RuntimeError(
            "Interpolation weights do not sum to one."
        )

    return (
        inside_target_indices,
        vertices,
        weights,
        coverage_mask,
    )


# ============================================================
# CHECK NEMO WEIGHTS AGAINST NaN REGION
# ============================================================

def check_nemo_weights_against_coverage(
    weights_file,
    coverage_mask,
    nav_lon,
    nav_lat,
):
    """
    Check whether the existing HRDPS-1km -> NEMO weights file
    references HRDPS 1 km grid points that will be NaN.

    HRDPS source indices in the weights file are assumed to be:

        src01
        src02
        src03
        src04

    with corresponding:

        wgt01
        wgt02
        wgt03
        wgt04

    Source indices are 1-based flattened indices with x changing
    fastest:

        index = x + nx * y + 1

    where Python y/x are zero-based.

    IMPORTANT:
    Every srcXX index is checked, even where wgtXX == 0.
    """

    print()
    print("=" * 78)
    print("CHECKING NEMO WEIGHTS AGAINST 2.5 km COVERAGE")
    print("=" * 78)

    print(
        f"Weights file:\n{weights_file}"
    )

    if not weights_file.exists():

        print()
        print(
            "WARNING: weights file does not exist."
        )

        print(
            "Skipping NaN-reference check."
        )

        return

    ny, nx = coverage_mask.shape

    n_hrdps_points = ny * nx

    coverage_flat = (
        coverage_mask.ravel()
    )

    lon_flat = np.asarray(
        nav_lon
    ).ravel()

    lat_flat = np.asarray(
        nav_lat
    ).ravel()

    src_names = [
        "src01",
        "src02",
        "src03",
        "src04",
    ]

    wgt_names = [
        "wgt01",
        "wgt02",
        "wgt03",
        "wgt04",
    ]

    total_stencil_references = 0

    illegal_index_references = 0

    nan_region_references = 0

    nonzero_nan_references = 0

    bad_source_indices = set()

    bad_examples = []

    target_bad_any = None

    target_bad_nonzero = None

    weight_shape = None

    weight_dimensions = None

    with Dataset(
        weights_file,
        "r",
    ) as ds:

        # ----------------------------------------------------
        # Required variables
        # ----------------------------------------------------

        missing = [
            name
            for name in src_names + wgt_names
            if name not in ds.variables
        ]

        if missing:
            raise KeyError(
                "Weights file is missing variables: "
                + ", ".join(missing)
            )

        # ----------------------------------------------------
        # Loop over the four source points
        # ----------------------------------------------------

        for slot, (src_name, wgt_name) in enumerate(
            zip(src_names, wgt_names),
            start=1,
        ):

            src_var = ds.variables[src_name]
            wgt_var = ds.variables[wgt_name]

            src = src_var[:]
            wgt = wgt_var[:]

            if np.ma.isMaskedArray(src):
                src = np.ma.filled(
                    src,
                    -999999999,
                )

            if np.ma.isMaskedArray(wgt):
                wgt = np.ma.filled(
                    wgt,
                    np.nan,
                )

            src = np.asarray(
                src,
                dtype=np.int64,
            )

            wgt = np.asarray(
                wgt,
                dtype=np.float64,
            )

            if weight_shape is None:

                weight_shape = src.shape

                weight_dimensions = (
                    src_var.dimensions
                )

                target_bad_any = np.zeros(
                    src.shape,
                    dtype=bool,
                )

                target_bad_nonzero = np.zeros(
                    src.shape,
                    dtype=bool,
                )

            elif src.shape != weight_shape:

                raise ValueError(
                    f"{src_name} shape {src.shape} "
                    f"does not match {weight_shape}"
                )

            total_stencil_references += (
                src.size
            )

            # -----------------------------------------------
            # Check 1-based source-index legality.
            # -----------------------------------------------

            valid_index = (
                (src >= 1)
                & (src <= n_hrdps_points)
            )

            invalid_index = (
                ~valid_index
            )

            n_invalid = np.count_nonzero(
                invalid_index
            )

            illegal_index_references += (
                n_invalid
            )

            target_bad_any |= (
                invalid_index
            )

            if n_invalid:

                flat_bad = np.flatnonzero(
                    invalid_index.ravel()
                )

                for pos in flat_bad:

                    if len(bad_examples) >= MAX_BAD_EXAMPLES:
                        break

                    target_position = np.unravel_index(
                        pos,
                        src.shape,
                    )

                    src_value = (
                        src.ravel()[pos]
                    )

                    weight_value = (
                        wgt.ravel()[pos]
                    )

                    bad_examples.append(
                        {
                            "slot": slot,
                            "target": target_position,
                            "src": int(src_value),
                            "weight": float(weight_value),
                            "reason": "illegal source index",
                            "y": None,
                            "x": None,
                            "lon": None,
                            "lat": None,
                        }
                    )

            # -----------------------------------------------
            # Check coverage for legal source indices.
            # -----------------------------------------------

            valid_positions = np.flatnonzero(
                valid_index.ravel()
            )

            src_flat = src.ravel()
            wgt_flat = wgt.ravel()

            # Convert Fortran-style 1-based flattened index
            # to Python zero-based flattened index.
            source_idx0 = (
                src_flat[valid_positions]
                - 1
            )

            is_covered = coverage_flat[
                source_idx0
            ]

            bad_local = (
                ~is_covered
            )

            if np.any(bad_local):

                bad_positions = (
                    valid_positions[bad_local]
                )

                bad_idx0 = (
                    src_flat[bad_positions]
                    - 1
                )

                bad_weights = (
                    wgt_flat[bad_positions]
                )

                nan_region_references += (
                    len(bad_positions)
                )

                active = (
                    np.isfinite(bad_weights)
                    & (
                        np.abs(bad_weights)
                        > WEIGHT_EPS
                    )
                )

                nonzero_nan_references += (
                    np.count_nonzero(active)
                )

                bad_mask_this_slot = (
                    np.zeros(
                        src.size,
                        dtype=bool,
                    )
                )

                bad_mask_this_slot[
                    bad_positions
                ] = True

                bad_mask_this_slot = (
                    bad_mask_this_slot.reshape(
                        src.shape
                    )
                )

                target_bad_any |= (
                    bad_mask_this_slot
                )

                active_mask = (
                    np.zeros(
                        src.size,
                        dtype=bool,
                    )
                )

                active_mask[
                    bad_positions[active]
                ] = True

                active_mask = (
                    active_mask.reshape(
                        src.shape
                    )
                )

                target_bad_nonzero |= (
                    active_mask
                )

                for source_point in np.unique(
                    bad_idx0
                ):
                    bad_source_indices.add(
                        int(source_point)
                    )

                # -------------------------------------------
                # Save human-readable examples.
                # -------------------------------------------

                for pos in bad_positions:

                    if len(bad_examples) >= MAX_BAD_EXAMPLES:
                        break

                    idx0 = (
                        src_flat[pos]
                        - 1
                    )

                    sy = idx0 // nx
                    sx = idx0 % nx

                    target_position = (
                        np.unravel_index(
                            pos,
                            src.shape,
                        )
                    )

                    bad_examples.append(
                        {
                            "slot": slot,
                            "target": target_position,
                            "src": int(
                                src_flat[pos]
                            ),
                            "weight": float(
                                wgt_flat[pos]
                            ),
                            "reason": "HRDPS point will be NaN",
                            "y": int(sy),
                            "x": int(sx),
                            "lon": float(
                                lon_flat[idx0]
                            ),
                            "lat": float(
                                lat_flat[idx0]
                            ),
                        }
                    )

    # ========================================================
    # REPORT
    # ========================================================

    n_bad_targets = np.count_nonzero(
        target_bad_any
    )

    n_active_bad_targets = np.count_nonzero(
        target_bad_nonzero
    )

    print()
    print(
        f"HRDPS 1 km grid size       : "
        f"{ny} x {nx} = {n_hrdps_points:,}"
    )

    print(
        f"Weights array dimensions   : "
        f"{weight_dimensions}"
    )

    print(
        f"Weights array shape        : "
        f"{weight_shape}"
    )

    print(
        f"Total stencil references   : "
        f"{total_stencil_references:,}"
    )

    print(
        f"Illegal source indices      : "
        f"{illegal_index_references:,}"
    )

    print(
        f"References into NaN region : "
        f"{nan_region_references:,}"
    )

    print(
        f"Distinct NaN source points : "
        f"{len(bad_source_indices):,}"
    )

    print(
        f"NEMO targets touching NaN  : "
        f"{n_bad_targets:,}"
    )

    print(
        f"Nonzero-weight NaN refs     : "
        f"{nonzero_nan_references:,}"
    )

    print(
        f"NEMO targets with nonzero "
        f"NaN weight: "
        f"{n_active_bad_targets:,}"
    )

    # --------------------------------------------------------
    # Clear safety message
    # --------------------------------------------------------

    if (
        illegal_index_references == 0
        and nan_region_references == 0
    ):

        print()
        print(
            "OK: the NEMO weights file does NOT reference "
            "any HRDPS 1 km point in the NaN region."
        )

        print(
            "The smaller 2.5 km atmospheric domain still "
            "covers every HRDPS point used by the NEMO stencil."
        )

        return

    print()
    print("!" * 78)
    print("WARNING")
    print("!" * 78)

    if illegal_index_references:

        print(
            f"The weights file contains "
            f"{illegal_index_references:,} illegal "
            f"HRDPS source-index references."
        )

    if nan_region_references:

        print(
            f"The NEMO weights stencil references "
            f"{nan_region_references:,} HRDPS 1 km locations "
            f"that will contain NaN."
        )

        if nonzero_nan_references:

            print(
                f"Among these, "
                f"{nonzero_nan_references:,} have a "
                f"NON-ZERO interpolation weight."
            )

        else:

            print(
                "All of these currently have zero weights, "
                "but they are still reported because "
                "0 * NaN is NaN under IEEE arithmetic."
            )

    # --------------------------------------------------------
    # Print examples
    # --------------------------------------------------------

    if bad_examples:

        print()
        print(
            f"First {len(bad_examples)} problematic "
            f"stencil references:"
        )

        print("-" * 78)

        for example in bad_examples:

            target_str = str(
                example["target"]
            )

            print(
                f"src{example['slot']:02d}"
                f"{target_str}: "
                f"src_index={example['src']}, "
                f"weight={example['weight']:.16g}"
            )

            print(
                f"    reason: "
                f"{example['reason']}"
            )

            if example["y"] is not None:

                print(
                    f"    HRDPS 1km y={example['y']}, "
                    f"x={example['x']}, "
                    f"lon={example['lon']:.8f}, "
                    f"lat={example['lat']:.8f}"
                )

        if (
            illegal_index_references
            + nan_region_references
            > len(bad_examples)
        ):

            print()
            print(
                "... additional problematic references "
                "were not printed."
            )

    print("!" * 78)


# ============================================================
# APPLY INTERPOLATION TO ONE 2-D FIELD
# ============================================================

def interpolate_field(
    src_field,
    target_shape,
    inside_target_indices,
    vertices,
    weights,
):
    """
    Interpolate one 2-D field.

    Outside the 2.5 km convex hull remains genuine IEEE NaN.
    """

    if np.ma.isMaskedArray(
        src_field
    ):
        src_field = np.ma.filled(
            src_field,
            np.nan,
        )

    src_field = np.asarray(
        src_field,
        dtype=np.float64,
    )

    flat_src = src_field.ravel()

    # Three source values for every covered target point.
    values = flat_src[
        vertices
    ]

    finite = np.isfinite(
        values
    )

    # If one of the three source values happens to be missing,
    # ignore it and renormalize the remaining finite weights.
    effective_weights = (
        weights * finite
    )

    weight_sum = effective_weights.sum(
        axis=1
    )

    weighted_sum = np.sum(
        np.where(
            finite,
            values,
            0.0,
        )
        * effective_weights,
        axis=1,
    )

    interpolated = np.full(
        len(inside_target_indices),
        np.nan,
        dtype=np.float64,
    )

    good = weight_sum > 0.0

    interpolated[good] = (
        weighted_sum[good]
        / weight_sum[good]
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # The ENTIRE target grid starts as NaN.
    #
    # Only points inside the 2.5 km convex hull are populated.
    # --------------------------------------------------------

    output = np.full(
        np.prod(target_shape),
        np.nan,
        dtype=np.float64,
    )

    output[
        inside_target_indices
    ] = interpolated

    return output.reshape(
        target_shape
    )


# ============================================================
# WRITE ONE DAILY FILE
# ============================================================

def write_one_day(
    source_file,
    output_file,
    reference,
    inside_target_indices,
    vertices,
    weights,
):
    """Interpolate one complete 2.5 km daily file."""

    print()
    print("=" * 78)

    print(
        f"SOURCE : {source_file}"
    )

    print(
        f"OUTPUT : {output_file}"
    )

    print("=" * 78)

    if output_file.exists():

        if not OVERWRITE:

            raise FileExistsError(
                f"\nOutput already exists:\n"
                f"{output_file}\n\n"
                "Set OVERWRITE = True if you really "
                "want to replace it."
            )

        output_file.unlink()

    ny = len(
        reference.dimensions["y"]
    )

    nx = len(
        reference.dimensions["x"]
    )

    target_shape = (
        ny,
        nx,
    )

    with Dataset(
        source_file,
        "r",
    ) as src, Dataset(
        output_file,
        "w",
        format="NETCDF4",
    ) as dst:

        # ====================================================
        # SANITY CHECK SOURCE VARIABLES
        # ====================================================

        for name in VARIABLES:

            if name not in src.variables:

                raise KeyError(
                    f"{name!r} is missing from "
                    f"{source_file}"
                )

            dims = (
                src.variables[name].dimensions
            )

            if dims != (
                "time_counter",
                "y",
                "x",
            ):

                raise ValueError(
                    f"{name}: unexpected dimensions "
                    f"{dims}; expected "
                    "('time_counter', 'y', 'x')"
                )

        nt = len(
            src.dimensions["time_counter"]
        )

        print(
            f"time records : {nt}"
        )

        print(
            f"target grid  : {ny} x {nx}"
        )

        # ====================================================
        # DIMENSIONS
        # ====================================================

        # Unlimited time dimension.
        dst.createDimension(
            "time_counter",
            None,
        )

        dst.createDimension(
            "y",
            ny,
        )

        dst.createDimension(
            "x",
            nx,
        )

        # ====================================================
        # GLOBAL ATTRIBUTES
        # ====================================================

        for attr in reference.ncattrs():

            if attr == "_NCProperties":
                continue

            try:
                dst.setncattr(
                    attr,
                    reference.getncattr(attr),
                )

            except Exception:
                pass

        dst.setncattr(
            "regridding_source",
            str(source_file),
        )

        dst.setncattr(
            "regridding_target_grid",
            str(REFERENCE_1KM),
        )

        dst.setncattr(
            "regridding_method",
            "Delaunay barycentric linear interpolation",
        )

        dst.setncattr(
            "regridding_extrapolation",
            "None. Target points outside the source "
            "convex hull contain IEEE NaN.",
        )

        dst.setncattr(
            "regridding_note",
            "Temporary HRDPS 1 km atmospheric forcing "
            "reconstructed from continental HRDPS "
            "2.5 km forcing.",
        )

        old_history = getattr(
            dst,
            "history",
            "",
        )

        new_history = (
            f"{datetime.now().isoformat(timespec='seconds')}: "
            "interpolated HRDPS continental 2.5 km forcing "
            "onto HRDPS 1 km grid; no extrapolation"
        )

        if old_history:

            dst.setncattr(
                "history",
                old_history
                + "\n"
                + new_history,
            )

        else:

            dst.setncattr(
                "history",
                new_history,
            )

        # ====================================================
        # x / y
        # ====================================================

        for coord_name in [
            "x",
            "y",
        ]:

            if coord_name in reference.variables:

                ref_var = (
                    reference.variables[
                        coord_name
                    ]
                )

                fill_value = get_fill_value(
                    ref_var
                )

                kwargs = {}

                if fill_value is not None:
                    kwargs["fill_value"] = fill_value

                out_var = dst.createVariable(
                    coord_name,
                    ref_var.datatype,
                    ref_var.dimensions,
                    **kwargs,
                )

                copy_nc_attrs(
                    ref_var,
                    out_var,
                )

                out_var[:] = ref_var[:]

            else:

                dim_length = len(
                    reference.dimensions[
                        coord_name
                    ]
                )

                out_var = dst.createVariable(
                    coord_name,
                    "i4",
                    (coord_name,),
                )

                out_var[:] = np.arange(
                    dim_length
                )

        # ====================================================
        # nav_lon / nav_lat FROM REAL 1 km FILE
        # ====================================================

        for coord_name in [
            "nav_lon",
            "nav_lat",
        ]:

            ref_var = (
                reference.variables[
                    coord_name
                ]
            )

            fill_value = get_fill_value(
                ref_var
            )

            kwargs = {}

            if fill_value is not None:
                kwargs["fill_value"] = fill_value

            out_var = dst.createVariable(
                coord_name,
                ref_var.datatype,
                ("y", "x"),
                zlib=True,
                complevel=COMPRESSION_LEVEL,
                shuffle=True,
                **kwargs,
            )

            copy_nc_attrs(
                ref_var,
                out_var,
            )

            out_var[:, :] = ref_var[:, :]

        # ====================================================
        # time_counter FROM 2.5 km FILE
        #
        # NO TIME SHIFT
        # NO TIME INTERPOLATION
        # ====================================================

        src_time = (
            src.variables[
                "time_counter"
            ]
        )

        time_fill = get_fill_value(
            src_time
        )

        kwargs = {}

        if time_fill is not None:
            kwargs["fill_value"] = (
                time_fill
            )

        dst_time = dst.createVariable(
            "time_counter",
            src_time.datatype,
            ("time_counter",),
            **kwargs,
        )

        copy_nc_attrs(
            src_time,
            dst_time,
            skip={
                "scale_factor",
                "add_offset",
            },
        )

        dst_time[:] = src_time[:]

        # ====================================================
        # STATIC AUXILIARY VARIABLES FROM REFERENCE
        # ====================================================

        already_written = {
            "time_counter",
            "x",
            "y",
            "nav_lon",
            "nav_lat",
            *VARIABLES,
        }

        for name, ref_var in reference.variables.items():

            if name in already_written:
                continue

            if (
                "time_counter"
                in ref_var.dimensions
            ):
                continue

            if not all(
                dim in dst.dimensions
                for dim in ref_var.dimensions
            ):
                continue

            fill_value = get_fill_value(
                ref_var
            )

            kwargs = {}

            if fill_value is not None:
                kwargs["fill_value"] = (
                    fill_value
                )

            try:

                out_var = dst.createVariable(
                    name,
                    ref_var.datatype,
                    ref_var.dimensions,
                    **kwargs,
                )

                copy_nc_attrs(
                    ref_var,
                    out_var,
                )

                out_var[:] = ref_var[:]

                print(
                    "Copied static auxiliary "
                    f"variable: {name}"
                )

            except Exception as exc:

                print(
                    "WARNING: skipped auxiliary "
                    f"variable {name}: {exc}"
                )

        # ====================================================
        # ATMOSPHERIC VARIABLES
        # ====================================================

        for name in VARIABLES:

            print()
            print("-" * 78)
            print(
                f"Interpolating: {name}"
            )

            src_var = (
                src.variables[name]
            )

            if not np.issubdtype(
                np.dtype(src_var.datatype),
                np.floating,
            ):

                raise TypeError(
                    f"{name} is not floating point; "
                    "cannot store NaN safely."
                )

            # Prefer metadata from genuine 1 km forcing.
            if name in reference.variables:

                metadata_var = (
                    reference.variables[name]
                )

            else:

                metadata_var = src_var

            fill_value = get_fill_value(
                metadata_var
            )

            if fill_value is None:

                fill_value = get_fill_value(
                    src_var
                )

            create_kwargs = {
                "zlib": True,
                "complevel": COMPRESSION_LEVEL,
                "shuffle": True,
                "chunksizes": (
                    1,
                    min(256, ny),
                    min(256, nx),
                ),
            }

            if fill_value is not None:

                create_kwargs[
                    "fill_value"
                ] = fill_value

            dst_var = dst.createVariable(
                name,
                src_var.datatype,
                (
                    "time_counter",
                    "y",
                    "x",
                ),
                **create_kwargs,
            )

            # Source attributes first.
            copy_nc_attrs(
                src_var,
                dst_var,
                skip={
                    "scale_factor",
                    "add_offset",
                },
            )

            # Genuine 1 km metadata takes precedence.
            if name in reference.variables:

                copy_nc_attrs(
                    reference.variables[name],
                    dst_var,
                    skip={
                        "scale_factor",
                        "add_offset",
                    },
                )

            overall_min = np.inf
            overall_max = -np.inf

            for t in range(nt):

                src_2d = src_var[
                    t,
                    :,
                    :,
                ]

                out_2d = interpolate_field(
                    src_2d,
                    target_shape,
                    inside_target_indices,
                    vertices,
                    weights,
                )

                finite = np.isfinite(
                    out_2d
                )

                if np.any(finite):

                    overall_min = min(
                        overall_min,
                        np.nanmin(
                            out_2d
                        ),
                    )

                    overall_max = max(
                        overall_max,
                        np.nanmax(
                            out_2d
                        ),
                    )

                # --------------------------------------------
                # IMPORTANT:
                #
                # Do NOT turn NaNs into a masked array.
                #
                # Write actual IEEE NaNs into the NetCDF file.
                # --------------------------------------------

                dst_var[
                    t,
                    :,
                    :,
                ] = out_2d.astype(
                    src_var.datatype
                )

                print(
                    f"\r  hour "
                    f"{t + 1:02d}/{nt:02d}",
                    end="",
                    flush=True,
                )

            print()

            if np.isfinite(
                overall_min
            ):

                print(
                    f"  finite output range: "
                    f"{overall_min:.8g} ... "
                    f"{overall_max:.8g}"
                )

            else:

                print(
                    "  WARNING: entire output "
                    "variable is NaN."
                )

        print()
        print(
            "Finished writing:"
        )

        print(
            output_file
        )


# ============================================================
# MAIN
# ============================================================

def main():

    if len(DATES) == 0:
        raise ValueError(
            "DATES is empty."
        )

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not REFERENCE_1KM.exists():

        raise FileNotFoundError(
            f"1 km reference file not found:\n"
            f"{REFERENCE_1KM}"
        )

    print("=" * 78)
    print(
        "HRDPS 2.5 km -> HRDPS 1 km"
    )
    print(
        "NO EXTRAPOLATION VERSION"
    )
    print("=" * 78)

    # ========================================================
    # OPEN THE REAL 1 km GRID
    # ========================================================

    with Dataset(
        REFERENCE_1KM,
        "r",
    ) as reference:

        target_lon = np.asarray(
            reference.variables[
                "nav_lon"
            ][:, :],
            dtype=np.float64,
        )

        target_lat = np.asarray(
            reference.variables[
                "nav_lat"
            ][:, :],
            dtype=np.float64,
        )

        print(
            f"Target 1 km grid: "
            f"{target_lon.shape}"
        )

        # ====================================================
        # FIRST 2.5 km FILE DEFINES SOURCE GRID
        # ====================================================

        first_source = make_source_filename(
            DATES[0]
        )

        if not first_source.exists():

            raise FileNotFoundError(
                f"2.5 km source file not found:\n"
                f"{first_source}"
            )

        with Dataset(
            first_source,
            "r",
        ) as src0:

            source_lon = np.asarray(
                src0.variables[
                    "nav_lon"
                ][:, :],
                dtype=np.float64,
            )

            source_lat = np.asarray(
                src0.variables[
                    "nav_lat"
                ][:, :],
                dtype=np.float64,
            )

        print(
            f"Source 2.5 km grid: "
            f"{source_lon.shape}"
        )

        # ====================================================
        # BUILD INTERPOLATION GEOMETRY
        # ====================================================

        (
            inside_target_indices,
            vertices,
            interpolation_weights,
            coverage_mask,
        ) = create_interpolation_geometry(
            source_lon,
            source_lat,
            target_lon,
            target_lat,
        )

        # ====================================================
        # CHECK EXISTING HRDPS-1km -> NEMO WEIGHTS
        #
        # This only needs to happen once because the NaN region
        # is purely determined by grid geometry.
        # ====================================================

        check_nemo_weights_against_coverage(
            NEMO_WEIGHTS_FILE,
            coverage_mask,
            target_lon,
            target_lat,
        )

        # ====================================================
        # PROCESS REQUESTED DAYS
        # ====================================================

        for date_string in DATES:

            source_file = (
                make_source_filename(
                    date_string
                )
            )

            output_file = (
                make_output_filename(
                    date_string
                )
            )

            if not source_file.exists():

                print()
                print(
                    f"WARNING: source file does "
                    f"not exist; skipping "
                    f"{date_string}"
                )

                print(
                    source_file
                )

                continue

            write_one_day(
                source_file,
                output_file,
                reference,
                inside_target_indices,
                vertices,
                interpolation_weights,
            )

    print()
    print("=" * 78)
    print("ALL DONE")
    print("=" * 78)


if __name__ == "__main__":
    main()

HRDPS 2.5 km -> HRDPS 1 km
NO EXTRAPOLATION VERSION
Target 1 km grid: (1180, 1330)
Source 2.5 km grid: (230, 190)

BUILDING 2.5 km -> 1 km INTERPOLATION GEOMETRY
Valid 2.5 km source points : 43,700
Valid 1 km target points   : 1,569,400

Building Delaunay triangulation...

Inside 2.5 km convex hull  : 246,127
Outside -> NaN             : 1,323,273
Invalid target coordinates : 0
NaN fraction               : 84.3171%

Barycentric-weight check:
  min(sum(weights)) = 1.000000000000000
  max(sum(weights)) = 1.000000000000000
  min(weight)       = 1.033369110170490e-06
  max(weight)       = 9.991491365889342e-01

CHECKING NEMO WEIGHTS AGAINST 2.5 km COVERAGE
Weights file:
/home/jqiu/analysis-junqi/Analysis_Atmospheric_Forcing/Analysis_weights/HRDPS_1km_Weights/weights-HRDPS-1km_202108.nc

HRDPS 1 km grid size       : 1180 x 1330 = 1,569,400
Weights array dimensions   : ('y', 'x')
Weights array shape        : (898, 398)
Total stencil references   : 1,429,616
Illegal source indices      : 0
Re